# E. Python Data Types

A type tells Python what a value represents and which operations make sense for it. In ECG work, a sample index is an integer, a calibrated voltage is usually a floating-point number, and a beat label is text. I check these distinctions before doing arithmetic.

## E.1 Common built-in types

| Type | Example | Use in this book |
|---|---|---|
| `int` | `360` | Sample count or index |
| `float` | `0.25` | Voltage or duration |
| `complex` | `1 + 2j` | Fourier coefficient |
| `bool` | `True` | A condition or quality flag |
| `str` | `"MLII"` | Channel name or record identifier |
| `list` | `[100, 105]` | Ordered, mutable collection |
| `tuple` | `(0, 360)` | Fixed collection, such as segment boundaries |
| `dict` | `{"fs": 360}` | Named metadata |
| `set` | `{"N", "V"}` | Distinct beat labels |
| `NoneType` | `None` | An absent value, distinct from zero |

Python names refer to objects. Assigning a new object to a name can change the type of value it refers to; it does not convert the previous object.

## E.2 Inspect and convert values

In [1]:
fs = 360
for value in [fs, 0.25, "N", True, 1 + 2j, None]:
    print(repr(value), "->", type(value).__name__)
print("Time at sample 180 (s):", 180 / fs)
print("Integer conversion:", int(2.9))

360 -> int
0.25 -> float
'N' -> str
True -> bool
(1+2j) -> complex
None -> NoneType
Time at sample 180 (s): 0.5
Integer conversion: 2


Division with `/` produces a floating-point result here. Converting a float with `int()` truncates toward zero; it does not round to the nearest integer. Decide how a time boundary should map to samples before converting it.

## E.3 NumPy dtypes and overflow

Python integers can grow as needed, subject to memory limits. NumPy integer arrays use fixed widths. A signed `int16` holds values from −32,768 to 32,767; arithmetic within that dtype can overflow. I convert raw integers to floating point before squaring or filtering. Converting the type does not apply a physical calibration: the header still determines how digital counts map to millivolts.

In [2]:
import numpy as np
raw = np.array([1000, -1000], dtype=np.int16)
floating = raw.astype(np.float64)
print("Stored dtype:", raw.dtype)
print("Storage (bytes):", raw.nbytes)
print("Squared after conversion:", floating ** 2)
print("int16 limits:", np.iinfo(np.int16).min, np.iinfo(np.int16).max)

Stored dtype: int16
Storage (bytes): 4
Squared after conversion: [1000000. 1000000.]
int16 limits: -32768 32767


## E.4 Missing values and numerical comparisons

`None` represents an absent Python object. `np.nan` is a floating-point missing or undefined value. pandas also supports `pd.NA`. None of these means the measured voltage was zero.

In [3]:
values = np.array([0.2, np.nan, 0.4])
print("Finite positions:", np.isfinite(values))
print("Mean of available values:", np.nanmean(values))
print("Exact comparison:", 0.1 + 0.2 == 0.3)
print("Tolerance comparison:", np.isclose(0.1 + 0.2, 0.3))

Finite positions: [ True False  True]
Mean of available values: 0.30000000000000004
Exact comparison: False
Tolerance comparison: True


Floating-point values approximate most decimal fractions. Use a tolerance appropriate to the calculation when comparing them. Ignoring missing values in a summary is different from repairing a missing stretch of ECG. Preserve sample positions; dropping samples and joining the remainder changes the time axis.

## Further reading

- [Python built-in types](https://docs.python.org/3/library/stdtypes.html)
- [NumPy data types](https://numpy.org/doc/stable/user/basics.types.html)
- Continue with [NumPy](g_numpy.ipynb) or [pandas](f_pandas.ipynb).